# Part 3c — Embedding Training: txt-only | lag1
**Bach et al. (2025) Appendix G**

- Modality : Text only (RoBERTa) — no image, no tabular
- Mode      : Lag-1 features
- Outputs   : 6 zip files in `predictions/txt/lag1/`

**A100 GPU required. ~30-60 min (faster, no image encoder).**

## ① Mount Drive

In [1]:
# Drive mount not needed for local execution
print('✅ Local mode')

✅ Local mode


## ② Setup Environment

Installs pytorch-widedeep to an isolated folder and patches the gensim
dependency conflict. No restart needed.

In [2]:
# Dependencies: pytorch-widedeep==1.7.0, scipy==1.13.1, torchmetrics
# Install manually: pip install pytorch-widedeep==1.7.0 scipy==1.13.1 torchmetrics
print('✅ Packages assumed installed')

# ── Inject fake gensim to block all C extension conflicts ─────────────────
# pytorch-widedeep imports gensim for its text utilities only
# We do not use those — SAINT only needs the tabular components
import sys
import types as _types
fake_utils = _types.ModuleType('gensim.utils')
fake_utils.tokenize = lambda text, *a, **kw: text.lower().split()
for mod_name in [
    'gensim', 'gensim.utils', 'gensim.parsing',
    'gensim.parsing.preprocessing', 'gensim.corpora',
    'gensim.matutils', 'gensim.interfaces',
    'gensim.models', 'gensim.similarities',
]:
    sys.modules[mod_name] = _types.ModuleType(mod_name)
sys.modules['gensim.utils'] = fake_utils
print('✅ gensim conflict bypassed')

✅ Packages assumed installed
✅ gensim conflict bypassed


In [3]:
import os
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from datetime import datetime
from transformers import AutoTokenizer, AutoModel, BeitModel, BeitImageProcessor
from pytorch_widedeep.models.tabular.transformers.saint import SAINT
from pytorch_widedeep.preprocessing.tab_preprocessor import TabPreprocessor

print(f'torch            : {torch.__version__}')
print(f'CUDA available   : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU              : {torch.cuda.get_device_name(0)}')
    print(f'VRAM             : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device           : {DEVICE}')
print('✅ All imports successful')

if not torch.cuda.is_available():
    raise RuntimeError('No GPU detected. Switch to A100 GPU runtime.')

/home/<your-rc-username>/.conda/envs/demand_modeling/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch            : 2.11.0+cu128
CUDA available   : True
GPU              : NVIDIA H200
VRAM             : 150.1 GB
Device           : cuda
✅ All imports successful


## ③ Config

In [4]:
import os
from pathlib import Path

# Detect project root by walking up from CWD to find 'data/' and 'code/' folders
_cwd = Path.cwd()
PROJECT_ROOT = _cwd
for _ in range(5):
    if (PROJECT_ROOT / 'data').is_dir() and (PROJECT_ROOT / 'code').is_dir():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent
else:
    raise RuntimeError("Cannot find project root (expected 'data/' and 'code/' folders)")

ROOT      = str(PROJECT_ROOT) + os.sep
DATA_DIR  = str(PROJECT_ROOT / 'data') + os.sep
IMG_DIR   = DATA_DIR + 'images' + os.sep
PRED_DIR  = DATA_DIR + 'predictions' + os.sep
SPLIT_DIR = DATA_DIR + 'amzn_shoes_monthly_diffs_ffill_fixed_splits' + os.sep

BATCH_SIZE   = 32
EPOCHS_LEVEL = 25
EPOCHS_DIFF  = 15
LR           = 2e-5
EMB_DIMS     = [128, 256]
MOD          = 4
WINDOW       = 28
MAX_PERIODS  = 53
SEED         = 42
ENCODER_DIM  = 768

TEXT_MODEL_NAME  = 'cardiffnlp/twitter-roberta-base'
IMAGE_MODEL_NAME = 'microsoft/beit-base-patch16-224'

TAB_COLS = [
    'RATING',
    'REVIEW_COUNT',
    'New Offer Count: Current',
    'Count of retrieved live offers: New, FBA',
    'Count of retrieved live offers: New, FBM',
    'Lightning Deals: Upcoming Deal',
    'Buy Box: Is FBA',
]
TAB_CONTINUOUS = TAB_COLS  # all treated as continuous

for subdir in [
    'txt/time_indipendent', 'txt/lag1',
    'txtimg/time_indipendent', 'txtimg/lag1',
]:
    os.makedirs(PRED_DIR + subdir, exist_ok=True)

TODAY = datetime.now().strftime('%Y-%m-%d')
torch.manual_seed(SEED)
print(f'Config ready | TODAY={TODAY}')
print(f'Pred dir: {PRED_DIR}')


# ── 2-hour-cap resilience knobs ───────────────────────────────────────────────
# RC gpu-interactive sessions are capped at ~2h. We checkpoint every epoch to the
# HOME filesystem (NOT /tmp, which is wiped between sessions), stop cleanly before
# the cap, and resume on relaunch. A run becomes: launch -> auto-stops -> relaunch.
CKPT_DIR      = DATA_DIR + 'checkpoints' + os.sep   # persistent (home fs)
WALLCLOCK_MIN = int(os.environ.get('WALLCLOCK_MIN', 110))  # stop cleanly after N min
PATIENCE      = int(os.environ.get('PATIENCE', 8))         # early-stopping patience (epochs)
os.makedirs(CKPT_DIR, exist_ok=True)
import time as _time
START_TIME = _time.time()   # wall-clock origin for the time-budget check

# ── LoRA (Low-Rank Adaptation) knobs ──────────────────────────────────────────
# Within-subcat splits have only ~130-630 train products. Fully fine-tuning ~200M
# encoder params would memorize/overfit; freezing would make every subcat's encoders
# identical (no real within-subcat adaptation -> no experiment). LoRA trains only
# small low-rank deltas on the attention q/v projections (~1% of params), matching
# capacity to sample size while still genuinely adapting per subcat.
# (Hu et al. 2021, "LoRA"; HuggingFace `peft`.)
USE_LORA     = os.environ.get('USE_LORA', '1') == '1'
LORA_R       = int(os.environ.get('LORA_R', 8))      # rank of the low-rank update
LORA_ALPHA   = int(os.environ.get('LORA_ALPHA', 16))
LORA_DROPOUT = float(os.environ.get('LORA_DROPOUT', 0.1))
LORA_TARGETS = ['query', 'value']  # RoBERTa & BEiT self-attention projection names
print(f'Resilience: ckpt={CKPT_DIR} | budget={WALLCLOCK_MIN}min | patience={PATIENCE}')
print(f'LoRA: USE_LORA={USE_LORA} r={LORA_R} alpha={LORA_ALPHA} dropout={LORA_DROPOUT}')


Config ready | TODAY=2026-04-14
Pred dir: /home/<your-rc-username>/demand-modeling-data-men-8-whole/data/predictions/


## ④ Load and Prepare Panel Data

In [5]:
df_train_raw = pd.read_parquet(SPLIT_DIR + 'train-00000-of-00001.parquet')
df_val_raw   = pd.read_parquet(SPLIT_DIR + 'validation-00000-of-00001.parquet')

def prepare_df(df):
    df = df[df['window'] == float(WINDOW)].copy()
    df = df.dropna(subset=['SALES_RANK', 'PRICE'])
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values(['ASIN', 'date']).reset_index(drop=True)
    df['date_t'] = df['date'].astype('category').cat.codes
    df = df[df['date_t'] <= MAX_PERIODS]
    df = df[df['date_t'] % MOD == 0]
    for col in TAB_COLS:
        if col in df.columns:
            df[col] = df[col].fillna(0).astype(float)
    return df.reset_index(drop=True)

df_train = prepare_df(df_train_raw)
df_val   = prepare_df(df_val_raw)

print(f'Train : {len(df_train):,} rows | {df_train["ASIN"].nunique():,} ASINs | {df_train["date_t"].nunique()} periods')
print(f'Val   : {len(df_val):,} rows | {df_val["ASIN"].nunique():,} ASINs | {df_val["date_t"].nunique()} periods')

Train : 14,406 rows | 1,029 ASINs | 14 periods
Val   : 14,448 rows | 1,032 ASINs | 14 periods


## ⑤ First Differences and Lag-1 Features

In [6]:
def add_diffs(df):
    df = df.sort_values(['ASIN', 'date']).copy()
    df['DELTA_SALES_RANK'] = df.groupby('ASIN')['SALES_RANK'].diff()
    df['DELTA_PRICE']      = df.groupby('ASIN')['PRICE'].diff()
    return df.dropna(subset=['DELTA_SALES_RANK', 'DELTA_PRICE']).reset_index(drop=True)

def add_lag1(df):
    df = df.sort_values(['ASIN', 'date']).copy()
    df['SALES_RANK_lag1'] = df.groupby('ASIN')['SALES_RANK'].shift(1)
    df['PRICE_lag1']      = df.groupby('ASIN')['PRICE'].shift(1)
    return df.dropna(subset=['SALES_RANK_lag1', 'PRICE_lag1']).reset_index(drop=True)

df_train_level     = df_train.copy()
df_val_level       = df_val.copy()
df_train_diff      = add_diffs(df_train)
df_val_diff        = add_diffs(df_val)
df_train_lag1      = add_lag1(df_train_level)
df_val_lag1        = add_lag1(df_val_level)
df_train_diff_lag1 = add_lag1(df_train_diff)
df_val_diff_lag1   = add_lag1(df_val_diff)

print(f'Level     train/val : {len(df_train_level):,} / {len(df_val_level):,}')
print(f'Diff      train/val : {len(df_train_diff):,} / {len(df_val_diff):,}')
print(f'Lag1      train/val : {len(df_train_lag1):,} / {len(df_val_lag1):,}')
print(f'Diff+Lag1 train/val : {len(df_train_diff_lag1):,} / {len(df_val_diff_lag1):,}')

Level     train/val : 14,406 / 14,448
Diff      train/val : 13,377 / 13,416
Lag1      train/val : 13,377 / 13,416
Diff+Lag1 train/val : 12,348 / 12,384


## ⑥ Load Encoders and Fit Tabular Preprocessor

In [7]:
print('Loading text tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)
print(f'  ✅ {TEXT_MODEL_NAME}')

print('Loading image processor...')
image_processor = BeitImageProcessor.from_pretrained(IMAGE_MODEL_NAME)
print(f'  ✅ {IMAGE_MODEL_NAME}')

print('Fitting tabular preprocessor...')
all_tab = pd.concat([df_train_level, df_val_level])[TAB_COLS].copy()
tab_preprocessor = TabPreprocessor(
    continuous_cols = TAB_COLS,
    cat_embed_cols  = None,
    scale           = True,
)
tab_preprocessor.fit(all_tab)

# pytorch-widedeep 1.7.0 uses cat_embed_input from column_idx directly
_cat_embed_input = getattr(tab_preprocessor, 'cat_embed_input', None)

_saint_test = SAINT(
    column_idx      = tab_preprocessor.column_idx,
    cat_embed_input = _cat_embed_input,
    continuous_cols = TAB_COLS,
    input_dim       = 32,
    n_heads         = 4,
    n_blocks        = 2,
)

_saint_test.eval()
_dummy = torch.tensor(
    tab_preprocessor.transform(
        pd.DataFrame(np.zeros((2, len(TAB_COLS))), columns=TAB_COLS)
    ), dtype=torch.float32
)
with torch.no_grad():
    SAINT_OUT_DIM = _saint_test(_dummy).shape[-1]
del _saint_test

print(f'  ✅ TabPreprocessor fitted | SAINT output dim: {SAINT_OUT_DIM}')

Loading text tokenizer...


/home/<your-rc-username>/.conda/envs/demand_modeling/lib/python3.11/site-packages/transformers/models/roberta/tokenization_roberta.py:144: DeprecationWarning: Deprecated in 0.9.0: BPE.__init__ will not create from files anymore, try `BPE.from_file` instead
  BPE(


  ✅ cardiffnlp/twitter-roberta-base
Loading image processor...
  ✅ microsoft/beit-base-patch16-224
Fitting tabular preprocessor...
  ✅ TabPreprocessor fitted | SAINT output dim: 224


/home/<your-rc-username>/.conda/envs/demand_modeling/lib/python3.11/site-packages/pytorch_widedeep/preprocessing/tab_preprocessor.py:299: DeprecationWarning: 'scale' and 'already_standard' will be deprecated in the next release. Please use 'cols_to_scale' instead
  self._check_inputs(cat_embed_cols)


## ⑦ Model Architecture (Appendix G, Figure 18)

In [8]:
class CrossAttentionBlock(nn.Module):
    def __init__(self, dim=768, num_heads=8, dropout=0.1):
        super().__init__()
        self.attn    = nn.MultiheadAttention(dim, num_heads, dropout=dropout, batch_first=True)
        self.norm    = nn.LayerNorm(dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, query, context):
        q  = query.unsqueeze(1)
        kv = context.unsqueeze(1)
        out, _ = self.attn(q, kv, kv)
        return self.norm(query + self.dropout(out.squeeze(1)))


class FullDemandModel(nn.Module):
    """
    Full multimodal embedding model — Bach et al. (2025) Appendix G.
    Three encoders fused via all-to-all Cross Attention Blocks.
    All parameters fine-tuned end-to-end.
    """
    def __init__(self, emb_dim, txt_only, use_lag, saint_out_dim, enc_dim=768):
        super().__init__()
        self.txt_only = txt_only
        self.use_lag  = use_lag

        # Pretrained encoders
        self.text_encoder = AutoModel.from_pretrained(TEXT_MODEL_NAME)
        if not txt_only:
            self.image_encoder = BeitModel.from_pretrained(IMAGE_MODEL_NAME)
            self.saint_encoder = SAINT(
                column_idx      = tab_preprocessor.column_idx,
                cat_embed_input = getattr(tab_preprocessor, 'cat_embed_input', None),
                continuous_cols = TAB_COLS,
                input_dim       = 32,
                n_heads         = 4,
                n_blocks        = 2,
            )
            self.tab_proj = (
                nn.Linear(saint_out_dim, enc_dim)
                if saint_out_dim != enc_dim else nn.Identity()
            )

        # Layer Norms
        self.norm_txt = nn.LayerNorm(enc_dim)
        if not txt_only:
            self.norm_img = nn.LayerNorm(enc_dim)
            self.norm_tab = nn.LayerNorm(enc_dim)
            # Cross Attention Blocks — all-to-all
            self.cab_txt  = CrossAttentionBlock(enc_dim)
            self.cab_img  = CrossAttentionBlock(enc_dim)
            self.cab_tab  = CrossAttentionBlock(enc_dim)
            fusion_dim = enc_dim * 3
        else:
            fusion_dim = enc_dim

        lag_dim = 2 if use_lag else 0

        # FC projection to embedding E
        self.projection = nn.Sequential(
            nn.Linear(fusion_dim + lag_dim, 1024),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(1024, emb_dim),
        )
        # Two output heads for DoubleML
        self.head_q = nn.Sequential(nn.Linear(emb_dim, 64), nn.GELU(), nn.Linear(64, 1))
        self.head_p = nn.Sequential(nn.Linear(emb_dim, 64), nn.GELU(), nn.Linear(64, 1))

    def forward(self, input_ids, attention_mask,
                pixel_values=None, tab_tensor=None, lag_feats=None):
        e_txt = self.norm_txt(
            self.text_encoder(input_ids=input_ids,
                              attention_mask=attention_mask).last_hidden_state[:, 0, :]
        )
        if not self.txt_only:
            e_img = self.norm_img(
                self.image_encoder(pixel_values=pixel_values).last_hidden_state[:, 0, :]
            )
            e_tab = self.norm_tab(
                self.tab_proj(self.saint_encoder(tab_tensor))
            )
            e_txt_f = self.cab_txt(e_txt, (e_img + e_tab) / 2)
            e_img_f = self.cab_img(e_img, (e_txt + e_tab) / 2)
            e_tab_f = self.cab_tab(e_tab, (e_txt + e_img) / 2)
            fused   = torch.cat([e_txt_f, e_img_f, e_tab_f], dim=-1)
        else:
            fused = e_txt

        if self.use_lag and lag_feats is not None:
            fused = torch.cat([fused, lag_feats], dim=-1)

        E     = self.projection(fused)
        q_hat = self.head_q(E).squeeze(-1)
        p_hat = self.head_p(E).squeeze(-1)
        return E, q_hat, p_hat


print('✅ Architecture defined')

✅ Architecture defined


## ⑦b Why LoRA, and how it is applied here

**The problem.** Each within-subcategory split has only ~130-630 training products,
while the two pretrained encoders (RoBERTa for text, BEiT for images) carry ~200M
parameters. Fine-tuning all of them on so few products would memorize the training set
(overfit), so the resulting "embeddings" would not generalize. The opposite extreme,
*freezing* the encoders, would make every subcategory share identical encoders, so a
"within-subcat" embedding would be indistinguishable from the whole-dataset one and the
experiment would test nothing.

**LoRA (Low-Rank Adaptation, Hu et al. 2021).** Instead of updating a weight matrix `W`,
LoRA freezes `W` and learns a small low-rank update `dW = B @ A` (rank `r << dim`) added
in parallel; only `A` and `B` train. We inject these adapters into the attention
**query** and **value** projections of both encoders (`r=8, alpha=16, dropout=0.1`), so
roughly **1%** of encoder parameters are trainable. The base encoder weights stay fixed
while the adapters let the encoders *genuinely adapt to each subcategory* without the
capacity to memorize ~130 products. Everything downstream of the encoders (the SAINT
tabular encoder, the cross-attention fusion blocks, the projection to E, and the two
DoubleML heads) trains normally.

**Why this is the right regularizer here:** it ties trainable capacity to sample size,
keeps real per-subcat adaptation, and makes different-sized subcats adapt to a comparable
degree (the regularization the advisor asked for).

**Comparability caveat.** The original whole-gender ("lazy") embeddings were *fully*
fine-tuned, whereas these within-subcat ("proper") embeddings are LoRA-fine-tuned, so
lazy-vs-proper is not a perfectly method-identical A/B. This is a deliberate, far milder
compromise than freezing (LoRA still gives genuine adaptation), and full fine-tuning on
130-630 products is simply not viable. We flag this when interpreting results.

In [ ]:
# LoRA application: inject low-rank adapters into the encoders' attention q/v
# projections and freeze the base encoder weights. Returns the same model object.
def apply_lora(model):
    if not USE_LORA:
        return model
    try:
        from peft import LoraConfig, inject_adapter_in_model
    except ImportError as e:
        raise ImportError(
            "USE_LORA=1 but `peft` is not installed. On the RC, activate the "
            "demand_modeling conda env and run `pip install peft`. "
            "(Set USE_LORA=0 to fall back to full fine-tuning, which is NOT "
            "recommended for the small within-subcat splits.)"
        ) from e

    cfg = LoraConfig(
        r              = LORA_R,
        lora_alpha     = LORA_ALPHA,
        lora_dropout   = LORA_DROPOUT,
        target_modules = LORA_TARGETS,   # 'query','value' match RoBERTa & BEiT attention
        bias           = 'none',
    )
    # Inject in place so each encoder keeps its class + forward signature (downstream
    # `.last_hidden_state` access and keyword args are unchanged).
    inject_adapter_in_model(cfg, model.text_encoder)
    if not model.txt_only:
        inject_adapter_in_model(cfg, model.image_encoder)

    # Freeze base encoder weights; train only the injected LoRA deltas. Everything
    # outside the encoders (SAINT, cross-attention, projection, heads) stays trainable.
    for name, p in model.named_parameters():
        if 'text_encoder' in name or 'image_encoder' in name:
            p.requires_grad = ('lora_' in name)
    return model


def trainable_report(model):
    tot = sum(p.numel() for p in model.parameters())
    tr  = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'    trainable {tr/1e6:.2f}M / {tot/1e6:.1f}M params ({100*tr/max(tot,1):.2f}%)')


print(f'\u2705 LoRA helper defined (USE_LORA={USE_LORA})')

## ⑧ Dataset and DataLoader

In [9]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image as PILImage

class ShoesDataset(Dataset):
    def __init__(self, df, target_q, target_p, use_lag=False):
        self.df       = df.reset_index(drop=True)
        self.target_q = target_q
        self.target_p = target_p
        self.use_lag  = use_lag
        self._tok_cache = {}
        self._img_cache = {}

    def _tok(self, asin, text):
        if asin not in self._tok_cache:
            enc = tokenizer(str(text), padding='max_length', truncation=True,
                            max_length=128, return_tensors='pt')
            self._tok_cache[asin] = {
                'input_ids':      enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0),
            }
        return self._tok_cache[asin]

    def _img(self, asin):
        if asin not in self._img_cache:
            path = IMG_DIR + f'{asin}.jpg'
            img  = PILImage.open(path).convert('RGB') if os.path.exists(path)                    else PILImage.new('RGB', (224, 224), (128, 128, 128))
            enc  = image_processor(images=img, return_tensors='pt')
            self._img_cache[asin] = enc['pixel_values'].squeeze(0)
        return self._img_cache[asin]

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        asin = row['ASIN']
        tok  = self._tok(asin, row['text'])
        tab  = tab_preprocessor.transform(
            pd.DataFrame([row[TAB_COLS].values.astype(float)], columns=TAB_COLS)
        ).squeeze(0)
        item = {
            'input_ids':      tok['input_ids'],
            'attention_mask': tok['attention_mask'],
            'pixel_values':   self._img(asin),
            'tab_tensor':     torch.tensor(tab, dtype=torch.float32),
            'q': torch.tensor(float(row[self.target_q]), dtype=torch.float32),
            'p': torch.tensor(float(row[self.target_p]),  dtype=torch.float32),
        }
        if self.use_lag:
            item['lag_feats'] = torch.tensor(
                [float(row['SALES_RANK_lag1']), float(row['PRICE_lag1'])],
                dtype=torch.float32
            )
        return item


def make_loader(df, shuffle, target_q='SALES_RANK', target_p='PRICE', use_lag=False):
    ds = ShoesDataset(df, target_q, target_p, use_lag)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=4, pin_memory=True)

print('✅ Dataset defined')

✅ Dataset defined


## ⑨ Training Utilities

In [10]:
import torch.optim as optim

def train_one_epoch(model, loader, optimizer, scaler, txt_only):
    model.train()
    total, n = 0.0, 0
    for batch in loader:
        ids  = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        pix  = batch['pixel_values'].to(DEVICE) if not txt_only else None
        tab  = batch['tab_tensor'].to(DEVICE)   if not txt_only else None
        lag  = batch['lag_feats'].to(DEVICE) if 'lag_feats' in batch else None
        q, p = batch['q'].to(DEVICE), batch['p'].to(DEVICE)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            _, q_hat, p_hat = model(ids, mask, pix, tab, lag)
            loss = nn.MSELoss()(q_hat, q) + nn.MSELoss()(p_hat, p)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        total += loss.item(); n += 1
    return total / n


@torch.no_grad()
def eval_loss(model, loader, txt_only):
    model.eval()
    total, n = 0.0, 0
    for batch in loader:
        ids  = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        pix  = batch['pixel_values'].to(DEVICE) if not txt_only else None
        tab  = batch['tab_tensor'].to(DEVICE)   if not txt_only else None
        lag  = batch['lag_feats'].to(DEVICE) if 'lag_feats' in batch else None
        q, p = batch['q'].to(DEVICE), batch['p'].to(DEVICE)
        with torch.amp.autocast('cuda'):
            _, q_hat, p_hat = model(ids, mask, pix, tab, lag)
            loss = nn.MSELoss()(q_hat, q) + nn.MSELoss()(p_hat, p)
        total += loss.item(); n += 1
    return total / n


@torch.no_grad()
def extract_and_save(model, df, txt_only, save_path, emb_dim):
    model.eval()
    target_q = 'DELTA_SALES_RANK' if 'DELTA_SALES_RANK' in df.columns else 'SALES_RANK'
    target_p = 'DELTA_PRICE'      if 'DELTA_PRICE'      in df.columns else 'PRICE'
    ds     = ShoesDataset(df, target_q, target_p, model.use_lag)
    loader = DataLoader(ds, batch_size=128, shuffle=False, num_workers=4)
    all_emb, all_q, all_p = [], [], []
    for batch in loader:
        ids  = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        pix  = batch['pixel_values'].to(DEVICE) if not txt_only else None
        tab  = batch['tab_tensor'].to(DEVICE)   if not txt_only else None
        lag  = batch['lag_feats'].to(DEVICE) if 'lag_feats' in batch else None
        with torch.amp.autocast('cuda'):
            emb, q_hat, p_hat = model(ids, mask, pix, tab, lag)
        all_emb.append(emb.cpu().float().numpy())
        all_q.append(q_hat.cpu().float().numpy())
        all_p.append(p_hat.cpu().float().numpy())
    emb_arr  = np.concatenate(all_emb)
    q_arr    = np.concatenate(all_q)
    p_arr    = np.concatenate(all_p)
    df_r     = df.reset_index(drop=True)
    df_out   = pd.DataFrame(emb_arr, columns=[str(i) for i in range(emb_dim)])
    df_out.insert(0, 'pred_ml_m', p_arr)
    df_out.insert(0, 'pred_ml_l', q_arr)
    df_out.insert(0, 'time',  df_r['date'].dt.strftime('%Y-%m-%d'))
    df_out.insert(0, 'index', df_r['ASIN'])
    csv_name = os.path.basename(save_path).replace('.zip', '.csv')
    df_out.to_csv(save_path,
                  compression=dict(method='zip', archive_name=csv_name),
                  index=False)
    mb = os.path.getsize(save_path) / 1e6
    print(f'    → {os.path.basename(save_path)}  ({mb:.1f} MB, {len(df_out):,} rows)')


print('✅ Training utilities defined')


# ════════════════════════════════════════════════════════════════════════════
# Resumable training orchestrator (survives the RC 2-hour gpu-interactive cap)
# ════════════════════════════════════════════════════════════════════════════
# Each "unit" (a level model at one emb_dim, or the diff model) checkpoints every
# epoch to CKPT_DIR on the home filesystem. If the session is killed (or hits the
# wall-clock budget), relaunching the notebook resumes that unit from the saved epoch;
# units whose output zips already exist are skipped entirely. So a long run becomes:
# launch -> auto-stops at the cap -> relaunch -> continues, with no lost work.
import glob, time

class TimeBudgetExceeded(Exception):
    """Raised when the wall-clock budget is hit; caller stops cleanly (state saved)."""
    pass

def _minutes_elapsed():
    return (time.time() - START_TIME) / 60.0

def _resume_path(tag):
    return CKPT_DIR + f'resume_{tag}.pt'

def _best_path(tag):
    return CKPT_DIR + f'best_{tag}.pt'

def unit_done(out_globs):
    """A unit is complete iff every expected output zip already exists on disk."""
    return all(len(glob.glob(g)) > 0 for g in out_globs)

def train_unit(tag, build_model, epochs, tr_loader, vl_loader, txt_only):
    """Train one model with per-epoch checkpointing, resume, early stopping and a
    wall-clock budget. Returns (model_with_best_weights, best_epoch, best_loss).
    Resumes from saved state if present; raises TimeBudgetExceeded if the budget is
    hit mid-run (after saving state) so the caller can stop the notebook cleanly."""
    model     = build_model()
    optimizer = optim.AdamW([p for p in model.parameters() if p.requires_grad],
                            lr=LR, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler    = torch.amp.GradScaler('cuda')

    start_epoch, best_loss, best_ep, since_improve = 1, float('inf'), 0, 0

    # Resume from a saved per-epoch state if one exists for this unit.
    rp = _resume_path(tag)
    if os.path.exists(rp):
        st = torch.load(rp, map_location=DEVICE)
        model.load_state_dict(st['model'])
        optimizer.load_state_dict(st['optimizer'])
        scheduler.load_state_dict(st['scheduler'])
        start_epoch   = st['epoch'] + 1
        best_loss     = st['best_loss']
        best_ep       = st['best_ep']
        since_improve = st['since_improve']
        print(f'  \u21bb resume [{tag}] from epoch {start_epoch} (best val={best_loss:.4f} @ {best_ep})')

    for epoch in range(start_epoch, epochs + 1):
        tr_l = train_one_epoch(model, tr_loader, optimizer, scaler, txt_only)
        vl_l = eval_loss(model, vl_loader, txt_only)
        scheduler.step()

        improved = vl_l < best_loss
        if improved:
            best_loss, best_ep, since_improve = vl_l, epoch, 0
            torch.save(model.state_dict(), _best_path(tag))
        else:
            since_improve += 1

        # Persist full per-epoch resume state so a killed session continues exactly.
        torch.save({'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
                    'scheduler': scheduler.state_dict(), 'epoch': epoch,
                    'best_loss': best_loss, 'best_ep': best_ep,
                    'since_improve': since_improve}, rp)

        if epoch % 5 == 0 or epoch == 1 or improved:
            print(f'    Epoch {epoch:>3} | train={tr_l:.4f} | val={vl_l:.4f}'
                  f'{"  *best" if improved else ""}')

        # Early stopping: tiny subcats overfit fast, so stop once val stalls.
        if since_improve >= PATIENCE:
            print(f'  \u23f9 early stop [{tag}] at epoch {epoch} (no val gain for {PATIENCE} epochs)')
            break

        # Graceful time-budget stop: never die mid-epoch; save state and bail out.
        if _minutes_elapsed() >= WALLCLOCK_MIN:
            print(f'  \u23f8 time budget ({WALLCLOCK_MIN} min) reached after epoch {epoch}; '
                  f'state saved -> relaunch to continue [{tag}]')
            raise TimeBudgetExceeded(tag)

    # Done (finished or early-stopped): load best weights and clear resume state.
    model.load_state_dict(torch.load(_best_path(tag), map_location=DEVICE))
    if os.path.exists(rp):
        os.remove(rp)
    return model, best_ep, best_loss

print('\u2705 Resumable training orchestrator defined')

✅ Training utilities defined


## ⑩ Main Training Loop (resumable)

Trains this notebook's variant: level models at each `emb_dim` in `EMB_DIMS` plus the first-difference model. Each model is a resumable *unit* (`train_unit`): it checkpoints every epoch, early-stops on stalled validation, and stops cleanly at the wall-clock budget. Units whose output zips already exist are skipped, so after a 2-hour cap you simply relaunch and it continues where it left off.

In [11]:
VARIANTS = [
    (True,  True,  'lag1', 'txt/lag1'),
]

# Stable filename globs let us skip a unit whose outputs already exist (idempotent
# reruns after a relaunch). The exact filename embeds the best epoch / val-loss, which
# we do not know until training finishes, so we match the stable prefix/suffix instead.
def _level_globs(subfolder, modality, lag_type, emb_dim):
    return [PRED_DIR + subfolder + f'/{split}_pred_model-*_{lag_type}_{modality}_mod4-*'
            f'_{modality}_dim={emb_dim}_proj_emb_step=.zip' for split in ('train', 'val')]

def _diff_globs(subfolder, modality, inv):
    return [PRED_DIR + subfolder + f'/pred_diff_shoes-diff-model-{modality}_*'
            f'_{modality}_{inv}_{split}.zip' for split in ('train', 'val')]

try:
    for txt_only, use_lag, lag_type, subfolder in VARIANTS:
        modality = 'txt' if txt_only else 'txtimg'
        print('\n' + '=' * 65)
        print(f'VARIANT : {modality} | {lag_type}')
        print('=' * 65)

        tr_lv = df_train_lag1      if use_lag else df_train_level
        vl_lv = df_val_lag1        if use_lag else df_val_level
        tr_df = df_train_diff_lag1 if use_lag else df_train_diff
        vl_df = df_val_diff_lag1   if use_lag else df_val_diff

        # ── Level models (one per emb_dim) ───────────────────────────────────
        for emb_dim in EMB_DIMS:
            if unit_done(_level_globs(subfolder, modality, lag_type, emb_dim)):
                print(f'  \u2713 skip Level emb_dim={emb_dim} (outputs already exist)')
                continue
            print(f'\n  Level | emb_dim={emb_dim}')
            def _build(_ed=emb_dim, _to=txt_only, _ul=use_lag):
                # Inject LoRA on CPU, THEN move the whole model to DEVICE, so base
                # weights and LoRA deltas land on the GPU together (device-safe across
                # peft versions).
                m = FullDemandModel(_ed, _to, _ul, SAINT_OUT_DIM)
                m = apply_lora(m).to(DEVICE); trainable_report(m)
                return m
            tag = f'{modality}_{lag_type}_level{emb_dim}'
            model, best_ep, best_loss = train_unit(
                tag, _build, EPOCHS_LEVEL,
                make_loader(tr_lv, True,  use_lag=use_lag),
                make_loader(vl_lv, False, use_lag=use_lag), txt_only)
            print(f'  Best: epoch={best_ep} | val_loss={best_loss:.4f}')
            e_s, l_s = str(best_ep).zfill(3), f'{best_loss:.4f}'
            for split, sdf in [('train', tr_lv), ('val', vl_lv)]:
                fname = (f'{split}_pred_model-{TODAY}-embdim=None_{lag_type}_{modality}_mod4-'
                         f'epoch={e_s}-val_combined_loss={l_s}_{modality}_dim={emb_dim}_proj_emb_step=.zip')
                extract_and_save(model, sdf, txt_only, PRED_DIR + subfolder + '/' + fname, emb_dim)
            del model; torch.cuda.empty_cache()

        # ── Diff model (emb_dim fixed at 128) ────────────────────────────────
        inv = 'invariant' if not use_lag else lag_type
        if unit_done(_diff_globs(subfolder, modality, inv)):
            print('  \u2713 skip Diff (outputs already exist)')
        else:
            print(f'\n  Diff | {lag_type}')
            def _build_d(_to=txt_only, _ul=use_lag):
                m = FullDemandModel(128, _to, _ul, SAINT_OUT_DIM)
                m = apply_lora(m).to(DEVICE); trainable_report(m)
                return m
            tag = f'{modality}_{lag_type}_diff'
            model_d, best_de, best_d = train_unit(
                tag, _build_d, EPOCHS_DIFF,
                make_loader(tr_df, True,  'DELTA_SALES_RANK', 'DELTA_PRICE', use_lag),
                make_loader(vl_df, False, 'DELTA_SALES_RANK', 'DELTA_PRICE', use_lag), txt_only)
            print(f'  Best: epoch={best_de} | val_loss={best_d:.4f}')
            e_s, l_s = str(best_de).zfill(3), f'{best_d:.4f}'
            for split, sdf in [('train', tr_df), ('val', vl_df)]:
                fname = (f'pred_diff_shoes-diff-model-{modality}_{TODAY}-embdim=768_'
                         f'lag1-epoch={e_s}-val_combined_loss={l_s}_{modality}_{inv}_{split}.zip')
                extract_and_save(model_d, sdf, txt_only, PRED_DIR + subfolder + '/' + fname, 128)
            del model_d; torch.cuda.empty_cache()

    print('\nPart 3c complete: txt/lag1 — next: 00_part3d_train_txt_time_ind.ipynb')

except TimeBudgetExceeded as _e:
    print('\n' + '#' * 65)
    print(f'# TIME BUDGET REACHED while training unit [{_e}].')
    print('# Per-epoch state was saved under data/checkpoints/. Completed units are')
    print('# skipped automatically on rerun. RELAUNCH this notebook to continue.')
    print('#' * 65)


VARIANT : txt | lag1

  Level | emb_dim=256


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 36176.60it/s]
RobertaModel LOAD REPORT from: cardiffnlp/twitter-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.decoder.bias      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


    Epoch   1 | train=16.0592 | val=1.4075
    Epoch   5 | train=0.3731 | val=0.5084
    Epoch  10 | train=0.2810 | val=0.3966
    Epoch  15 | train=0.2557 | val=0.2998
    Epoch  20 | train=0.2383 | val=0.3064
    Epoch  25 | train=0.2317 | val=0.2997
  Best: epoch=17 | val_loss=0.2980
    → train_pred_model-2026-04-14-embdim=None_lag1_txt_mod4-epoch=017-val_combined_loss=0.2980_txt_dim=256_proj_emb_step=.zip  (9.8 MB, 13,377 rows)
    → val_pred_model-2026-04-14-embdim=None_lag1_txt_mod4-epoch=017-val_combined_loss=0.2980_txt_dim=256_proj_emb_step=.zip  (9.9 MB, 13,416 rows)

  Diff | lag1


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 40407.94it/s]
RobertaModel LOAD REPORT from: cardiffnlp/twitter-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.decoder.bias      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


    Epoch   1 | train=0.2530 | val=0.2398
    Epoch   5 | train=0.2429 | val=0.2319
    Epoch  10 | train=0.2320 | val=0.2243
    Epoch  15 | train=0.2232 | val=0.2209
  Best: epoch=12 | val_loss=0.2209
    → pred_diff_shoes-diff-model-txt_2026-04-14-embdim=768_lag1-epoch=012-val_combined_loss=0.2209_txt_lag1_train.zip  (5.8 MB, 12,348 rows)
    → pred_diff_shoes-diff-model-txt_2026-04-14-embdim=768_lag1-epoch=012-val_combined_loss=0.2209_txt_lag1_val.zip  (5.8 MB, 12,384 rows)

Part 3c complete: txt/lag1 — next: 00_part3d_train_txt_time_ind.ipynb
